In [ ]:
# CELL 1
# Install dependencies

!pip install transformers
!pip install sentencepiece
!pip install torch
!pip install newspaper3k
!pip install lxml_html_clean
!pip install rouge_score

In [ ]:
# CELL 2
# Import libraries

import re
import torch

from newspaper import Article
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [ ]:
# CELL 3
# Load model and tokenizer
# Model: cahya/t5-base-indonesian-summarization-cased
MODEL_NAME = 'cahya/t5-base-indonesian-summarization-cased'

print(f'Loading model: {MODEL_NAME}')
print('Downloading on first run (~850MB)...')

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

print(f'\nModel loaded. Running on: {device}')

In [ ]:
# CELL 4
# Input article URL

url = '-'

In [ ]:
# CELL 5
# Download and parse article

article = Article(url, language='id')
article.download()
article.parse()

title = article.title
text  = article.text

print('TITLE:')
print(title)
print('\nARTICLE (first 1500 chars):')
print(text[:1500])
print(f'\nTotal characters : {len(text)}')
print(f'Total words      : {len(text.split())}')

In [ ]:
# CELL 6
# Text cleaning function

def clean_text(text: str) -> str:

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Remove lines that look like image captions (short, all-caps)
    lines = text.splitlines()
    lines = [
        line for line in lines
        if not (len(line.strip()) < 60 and line.strip().isupper())
    ]
    text = '\n'.join(lines)

    # Collapse excessive blank lines and spaces
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)

    return text.strip()

In [ ]:
# CELL 7
# Chunk splitter
def split_into_chunks(
    text: str,
    max_words: int = 400,
    overlap_words: int = 50
) -> list:

    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + max_words, len(words))
        chunks.append(' '.join(words[start:end]))

        if end == len(words):
            break

        start += max_words - overlap_words

    return chunks

In [ ]:
# CELL 8
# Core summarization function

WHITESPACE_HANDLER = lambda text: re.sub(
    r'\s*\n+\s*', ' . ', text.strip()
)

def summarize_chunk(
    text: str,
    max_length: int = 150,
    min_length: int = 40,
    num_beams: int = 10,
    repetition_penalty: float = 2.5,
    length_penalty: float = 1.0,
    no_repeat_ngram_size: int = 2,
    do_sample: bool = True,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.95
) -> str:

    # T5 requires 'summarize: ' task prefix
    prepared = 'summarize: ' + WHITESPACE_HANDLER(text)

    input_ids = tokenizer.encode(
        prepared,
        return_tensors='pt',
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_length=max_length,
            min_length=min_length,
            num_beams=num_beams,
            repetition_penalty=repetition_penalty,
            length_penalty=length_penalty,
            no_repeat_ngram_size=no_repeat_ngram_size,
            early_stopping=True,
            use_cache=True,
            do_sample=do_sample,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

In [ ]:
# CELL 9
# Hierarchical summarizer

def summarize_article(
    text: str,
    chunk_max_words: int = 400,
    chunk_overlap: int = 50,
    verbose: bool = True
) -> str:

    cleaned = clean_text(text)
    word_count = len(cleaned.split())

    if word_count <= chunk_max_words:
        if verbose:
            print(f'Short article ({word_count} words) — single-pass summarization.')
        return summarize_chunk(cleaned)

    chunks = split_into_chunks(cleaned, chunk_max_words, chunk_overlap)

    if verbose:
        print(f'Long article ({word_count} words) — split into {len(chunks)} chunks.')

    partial_summaries = []
    for i, chunk in enumerate(chunks):
        if verbose:
            print(f'  Summarizing chunk {i + 1}/{len(chunks)}...')
        partial = summarize_chunk(
            chunk,
            max_length=100,
            min_length=25
        )
        partial_summaries.append(partial)

    merged = ' '.join(partial_summaries)

    if verbose:
        print('  Running final summarization pass on merged chunks...')

    return summarize_chunk(
        merged,
        max_length=150,
        min_length=40
    )

In [ ]:
# CELL 10
# Run summarization

summary = summarize_article(text, verbose=True)

print('\n' + '=' * 60)
print('TITLE  :', title)
print('\nSUMMARY:')
print(summary)
print('=' * 60)

In [ ]:
# CELL 11
# Summary statistics

original_words = len(text.split())
summary_words  = len(summary.split())
compression    = (original_words - summary_words) / original_words * 100

print('STATISTICS')
print(f'Original words : {original_words}')
print(f'Summary words  : {summary_words}')
print(f'Compression    : {compression:.2f}%')

### Validation: ROUGE Scores

ROUGE mengukur overlap kata antara summary dan artikel asli. Untuk abstractive summarization, skor ROUGE yang **lebih rendah dari extractive adalah normal** — model menghasilkan kalimat baru, bukan menyalin. Fokus pada apakah summary terasa informatif dan kohesif secara kualitatif, bukan hanya angka ROUGE.

In [ ]:
# CELL 12
# Validation: ROUGE Scores

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

scores = scorer.score(target=text, prediction=summary)

print('ROUGE Scores (vs. original article):')
print(f'{"Metric":<10} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 44)
for key, value in scores.items():
    print(
        f'{key:<10} '
        f'{value.precision:>10.4f} '
        f'{value.recall:>10.4f} '
        f'{value.fmeasure:>10.4f}'
    )